# Phase 2: Data Preprocessing Pipeline (Training Set Only)
## CICIoT2023 Network Intrusion Detection
### Đề tài Tốt nghiệp: Hệ thống phát hiện tấn công mạng IoT dựa trên XAI

---

### 🎯 Mục tiêu & Nguyên tắc Thiết kế Pipeline Tiền Xử Lý
1. **Chống Rò rỉ Dữ liệu Tuyệt đối (Strict Leakage Prevention)**:
   - Toàn bộ tham số tiền xử lý (`RobustScaler`, `imputation_medians`, `LabelEncoder`, `variance_threshold`) chỉ được fit **duy nhất trên tập huấn luyện (`train`)**.
   - Đóng gói toàn bộ pipeline vào `models/preprocessor.joblib` để áp dụng bất biến lên tập validation/test sau này.
2. **Khử Trùng Lặp 2 Chiều (Bidirectional Deduplication)**:
   - Khử trùng lặp nội bộ train (tránh học vẹt và lặp mẫu dễ).
   - Khử trùng lặp nội bộ test và **chặn rò rỉ chéo train-test (cross-set leakage)** bằng cơ chế băm dòng (Row Hashing).
3. **Lấy Mẫu Phân Tầng Không Lệch (Stratified Sampling)**:
   - Tuyệt đối không dùng `nrows` (vì dữ liệu CICIoT2023 được lưu theo thứ tự đợt tấn công). Lấy mẫu luôn phân tầng `train_test_split(stratify=y)`.
4. **Kỹ thuật Nén Outlier Bằng Log1p cho Heavy-Tail Traffic Metrics**:
   - Áp dụng $\log(1 + x)$ trên cả các đặc trưng lưu lượng nặng (`Rate`, `Tot sum`, `flow_duration`, `IAT`) lẫn các đặc trưng tỷ lệ để triệt tiêu các gai nhọn $10^6 - 10^9$, bảo vệ không gian khoảng cách Euclidean của thuật toán $k$-NN trong SMOTE.
5. **Phân Cấp Nhãn Đa Tầng**:
   - Cấp 1: **34 Lớp Chi tiết** (33 cuộc tấn công + BenignTraffic).
   - Cấp 2: **9 Nhóm Tấn công** (8 nhóm tấn công + Benign).
   - Cấp 3: **Nhị phân** (0: Benign, 1: Attack).

In [1]:
# ==============================================================================
# Cấu hình Môi trường Google Colab & Điều hướng Lưu trữ trực tiếp trên Drive
# ==============================================================================
import os
import sys

# Kiểm tra nếu đang chạy trên Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    
    # 🎯 ĐIỀU HƯỚNG WORKING DIRECTORY TRỰC TIẾP VÀO THƯ MỤC GOOGLE DRIVE
    DRIVE_DIR = "/content/drive/MyDrive/do_an"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    os.chdir(DRIVE_DIR)
    
    # Tạo sẵn các thư mục con lưu trữ trực tiếp trên Google Drive
    os.makedirs("data", exist_ok=True)
    os.makedirs("models", exist_ok=True)
    os.makedirs("plots", exist_ok=True)
    
    print(f"🚀 [Google Colab] Đã chuyển working directory trực tiếp về: {os.getcwd()}")
    print("📁 Tất cả kết quả (data/, models/, plots/) sẽ được ghi TRỰC TIẾP lên Google Drive!")
else:
    os.makedirs("data", exist_ok=True)
    os.makedirs("models", exist_ok=True)
    os.makedirs("plots", exist_ok=True)
    print(f"💻 [Local] Đang chạy trên máy cục bộ. Working directory: {os.getcwd()}")


Mounted at /content/drive
🚀 [Google Colab] Đã chuyển working directory trực tiếp về: /content/drive/MyDrive/do_an
📁 Tất cả kết quả (data/, models/, plots/) sẽ được ghi TRỰC TIẾP lên Google Drive!


In [2]:
# ==============================================================================
# Cell 1: Khởi tạo Môi trường & Schema
# ==============================================================================
import os
import gc
import json
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

ATTACK_CATEGORY_MAPPING = {
    'DDoS-ICMP_Flood': 'DDoS', 'DDoS-UDP_Flood': 'DDoS', 'DDoS-TCP_Flood': 'DDoS',
    'DDoS-PSHACK_Flood': 'DDoS', 'DDoS-SYN_Flood': 'DDoS', 'DDoS-RSTFINFlood': 'DDoS',
    'DDoS-SynonymousIP_Flood': 'DDoS', 'DDoS-ICMP_Fragmentation': 'DDoS',
    'DDoS-UDP_Fragmentation': 'DDoS', 'DDoS-ACK_Fragmentation': 'DDoS',
    'DDoS-HTTP_Flood': 'DDoS', 'DDoS-SlowLoris': 'DDoS',
    'DoS-UDP_Flood': 'DoS', 'DoS-TCP_Flood': 'DoS', 'DoS-SYN_Flood': 'DoS', 'DoS-HTTP_Flood': 'DoS',
    'Mirai-greeth_flood': 'Mirai', 'Mirai-udpplain': 'Mirai', 'Mirai-greip_flood': 'Mirai',
    'Recon-HostDiscovery': 'Recon', 'Recon-OSScan': 'Recon', 'Recon-PortScan': 'Recon',
    'Recon-PingSweep': 'Recon', 'VulnerabilityScan': 'Recon',
    'MITM-ArpSpoofing': 'Spoofing', 'DNS_Spoofing': 'Spoofing',
    'BrowserHijacking': 'Web', 'CommandInjection': 'Web', 'SqlInjection': 'Web',
    'XSS': 'Web', 'Uploading_Attack': 'Web',
    'DictionaryBruteForce': 'BruteForce',
    'Backdoor_Malware': 'Malware',
    'BenignTraffic': 'Benign'
}

DTYPE_DICT = {
    'flow_duration': 'float32', 'Header_Length': 'float32', 'Protocol Type': 'float32',
    'Duration': 'float32', 'Rate': 'float32', 'Srate': 'float32', 'Drate': 'float32',
    'fin_flag_number': 'float32', 'syn_flag_number': 'float32', 'rst_flag_number': 'float32',
    'psh_flag_number': 'float32', 'ack_flag_number': 'float32', 'ece_flag_number': 'float32',
    'cwr_flag_number': 'float32', 'ack_count': 'float32', 'syn_count': 'float32',
    'fin_count': 'float32', 'urg_count': 'float32', 'rst_count': 'float32',
    'HTTP': 'float32', 'HTTPS': 'float32', 'DNS': 'float32', 'Telnet': 'float32',
    'SMTP': 'float32', 'SSH': 'float32', 'IRC': 'float32', 'TCP': 'float32',
    'UDP': 'float32', 'DHCP': 'float32', 'ARP': 'float32', 'ICMP': 'float32',
    'IPv': 'float32', 'LLC': 'float32', 'Tot sum': 'float32', 'Min': 'float32',
    'Max': 'float32', 'AVG': 'float32', 'Std': 'float32', 'Tot size': 'float32',
    'IAT': 'float32', 'Number': 'float32', 'Magnitue': 'float32', 'Radius': 'float32',
    'Covariance': 'float32', 'Variance': 'float32', 'Weight': 'float32',
    'label': 'object'
}
print(f"Đã khởi tạo schema với 34 lớp chi tiết và 9 nhóm tấn công.")

Đã khởi tạo schema với 34 lớp chi tiết và 9 nhóm tấn công.


## Bước 1: Nạp Dữ Liệu Huấn Luyện (Raw Train Set)
> 🛡️ **Lưu ý phòng thủ**: Không nạp file output (`X_train.parquet`) làm input để tránh tự đầu độc pipeline. Chỉ nạp file `train.csv` thô.

In [3]:
# ==============================================================================
# Cell 2: Tải Dữ liệu Huấn luyện & Dedup trước khi Subsample
# ==============================================================================
candidate_paths = [
    'train.csv',
    'data/train.csv',
    'plots/train.csv',
    '../train.csv',
    '/content/drive/MyDrive/do_an/train.csv'
]

data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path:
    print(f"Nạp dữ liệu từ: {data_path}")
    df = pd.read_csv(data_path, dtype=DTYPE_DICT)
    assert 'label' in df.columns, "File không chứa cột 'label' — bạn đang nạp nhầm file đã tiền xử lý!"
    # Dedup trước khi lấy mẫu
    n_dup = df.duplicated().sum()
    if n_dup > 0:
        print(f"Khử {n_dup:,} dòng trùng lặp trước khi lấy mẫu ({n_dup / len(df) * 100:.2f}%)...")
        df.drop_duplicates(inplace=True)
        df.reset_index(drop=True, inplace=True)
else:
    # Fallback to existing preprocessed training pkl if train.csv is stored on cloud
    print("train.csv không có cục bộ (đang ở Colab/Cloud). Nạp từ data/X_train.pkl...")
    X_tmp = joblib.load('data/X_train.pkl')
    y_tmp = joblib.load('data/y_train.pkl')
    df = X_tmp.copy()
    df['label'] = y_tmp['label_name']

print(f"Dữ liệu sẵn sàng: {df.shape[0]:,} dòng x {df.shape[1]} cột")

Nạp dữ liệu từ: train.csv
Khử 1,377,193 dòng trùng lặp trước khi lấy mẫu (25.08%)...
Dữ liệu sẵn sàng: 4,114,778 dòng x 47 cột


## Bước 2: Làm Sạch Missing / Infinite Values & Trích Xuất Row Hashes
Lưu trữ mã băm (Row Hashes) của tập Train để chặn rò rỉ dữ liệu chéo (Cross-Set Leakage) với tập Test.

In [4]:
# ==============================================================================
# Cell 3: Làm sạch Inf/NaN & Lưu Trữ Train Row Hashes
# ==============================================================================
numeric_cols = df.select_dtypes(include=['float32', 'float64']).columns.tolist()
feat_cols_raw = [c for c in df.columns if c != 'label']

# 1. Thay inf bằng NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 2. Tính Medians
imputation_medians = {col: float(df[col].median()) for col in numeric_cols}
for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(imputation_medians[col])

if df['label'].isnull().sum() > 0:
    df.dropna(subset=['label'], inplace=True)

# 3. Tính toán train row hashes (bảo vệ chống rò rỉ chéo)
print("Đang tính toán Row Hashes của tập Train...")
train_row_hashes = set(pd.util.hash_pandas_object(df[feat_cols_raw], index=False).astype(np.uint64))
print(f"Đã lập chỉ mục băm cho {len(train_row_hashes):,} dòng train duy nhất.")

Đang tính toán Row Hashes của tập Train...
Đã lập chỉ mục băm cho 4,114,778 dòng train duy nhất.


## Bước 3: Loại Bỏ Đặc Trưng Hằng Số (Task 2.2)
Loại bỏ đặc trưng có phương sai $\le 10^{-12}$.

In [5]:
# ==============================================================================
# Cell 4: Loại Bỏ Đặc Trưng Phương Sai Bằng Không (Task 2.2)
# ==============================================================================
feat_cols = [c for c in df.columns if c != 'label']
vars_ = df[feat_cols].var()
dropped_zero_var_cols = vars_[vars_ <= 1e-12].index.tolist()

if dropped_zero_var_cols:
    print(f"Loại bỏ {len(dropped_zero_var_cols)} đặc trưng hằng số: {dropped_zero_var_cols}")
    df.drop(columns=dropped_zero_var_cols, inplace=True)
else:
    print("Không có đặc trưng hằng số nào.")

Loại bỏ 2 đặc trưng hằng số: ['Telnet', 'IRC']


## Bước 4: Centralized Feature Engineering với Nén Log1p (Task 2.3)
Áp dụng $\log(1+x)$ trên các đặc trưng lưu lượng có đuôi dài (Heavy-tail metrics) và các tỷ lệ.

In [6]:
# ==============================================================================
# Cell 5: Centralized Feature Engineering (Dùng chung cho Train/Test)
# ==============================================================================
def engineer_features(data, medians=None):
    eps = 1e-6
    
    # 1. Nén heavy-tail raw columns
    skewed_cols = ['flow_duration', 'Duration', 'Rate', 'Srate', 'Drate', 'Tot sum', 'Tot size', 'IAT', 'Variance']
    for c in skewed_cols:
        if c in data.columns:
            data[c] = np.log1p(np.maximum(data[c], 0)).astype('float32')
    
    # 2. Ratios (nén bằng log1p clip)
    if 'Tot sum' in data.columns and 'Number' in data.columns:
        data['avg_packet_size'] = np.log1p(np.clip(data['Tot sum'] / (data['Number'] + eps), 0, 1e6)).astype('float32')
    if 'Srate' in data.columns and 'Rate' in data.columns:
        data['rate_ratio'] = np.log1p(np.clip(data['Srate'] / (data['Rate'] + eps), 0, 1e6)).astype('float32')
    if 'Drate' in data.columns and 'Rate' in data.columns:
        data['drate_ratio'] = np.log1p(np.clip(data['Drate'] / (data['Rate'] + eps), 0, 1e6)).astype('float32')
    if 'Duration' in data.columns and 'flow_duration' in data.columns:
        data['duration_ratio'] = np.log1p(np.clip(data['Duration'] / (data['flow_duration'] + eps), 0, 1e6)).astype('float32')
    if 'Header_Length' in data.columns and 'Tot size' in data.columns:
        data['header_to_payload_ratio'] = np.log1p(np.clip(data['Header_Length'] / (data['Tot size'] + eps), 0, 1e6)).astype('float32')

    # 3. Flag Aggregates
    flag_nums = [c for c in ['fin_flag_number', 'syn_flag_number', 'rst_flag_number', 
                             'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number'] if c in data.columns]
    if flag_nums:
        data['total_flags'] = data[flag_nums].sum(axis=1).astype('float32')

    flag_counts = [c for c in ['ack_count', 'syn_count', 'fin_count', 'urg_count', 'rst_count'] if c in data.columns]
    if flag_counts:
        data['total_counts'] = data[flag_counts].sum(axis=1).astype('float32')

    if 'syn_count' in data.columns and 'ack_count' in data.columns:
        data['syn_ack_ratio'] = np.log1p(np.clip(data['syn_count'] / (data['ack_count'] + eps), 0, 1e6)).astype('float32')
    if 'rst_count' in data.columns and 'syn_count' in data.columns:
        data['rst_syn_ratio'] = np.log1p(np.clip(data['rst_count'] / (data['syn_count'] + eps), 0, 1e6)).astype('float32')

    # 4. Protocols
    if 'HTTP' in data.columns and 'HTTPS' in data.columns:
        data['web_traffic'] = (data['HTTP'] + data['HTTPS']).astype('float32')
    trans = [c for c in ['TCP', 'UDP', 'ICMP'] if c in data.columns]
    if trans:
        data['transport_traffic'] = data[trans].sum(axis=1).astype('float32')
    mgmt = [c for c in ['DNS', 'DHCP', 'ARP'] if c in data.columns]
    if mgmt:
        data['network_mgmt'] = data[mgmt].sum(axis=1).astype('float32')
    remote = [c for c in ['Telnet', 'SSH'] if c in data.columns]
    if remote:
        data['remote_access'] = data[remote].sum(axis=1).astype('float32')
    if 'HTTPS' in data.columns and 'SSH' in data.columns:
        data['is_encrypted'] = (data['HTTPS'] + data['SSH']).astype('float32')

    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    if medians:
        for c, med in medians.items():
            if c in data.columns and data[c].isnull().any():
                data[c] = data[c].fillna(med)
    data.fillna(0, inplace=True)
    return data

df = engineer_features(df, medians=imputation_medians)
print(f"Đã tạo thành công các đặc trưng mới. Kích thước hiện tại: {df.shape[1]} cột.")

Đã tạo thành công các đặc trưng mới. Kích thước hiện tại: 59 cột.


## Bước 5: Phân Cấp Nhãn & Chuẩn Hóa RobustScaler (Task 2.4 & 2.5)

In [7]:
# ==============================================================================
# Cell 6: Phân cấp Nhãn & Chuẩn Hóa RobustScaler
# ==============================================================================
df['attack_category'] = df['label'].map(lambda x: ATTACK_CATEGORY_MAPPING.get(x, 'Other'))
df['is_attack'] = (df['label'] != 'BenignTraffic').astype('int8')

targets = ['label', 'attack_category', 'is_attack']
features = [c for c in df.columns if c not in targets]

X_train = df[features].copy()
y_train = df['label'].copy()
y_cat_train = df['attack_category'].copy()
y_bin_train = df['is_attack'].copy()

# Encode
le_label = LabelEncoder()
y_train_enc = le_label.fit_transform(y_train)

le_cat = LabelEncoder()
y_cat_train_enc = le_cat.fit_transform(y_cat_train)

label_mapping = {label: int(code) for code, label in enumerate(le_label.classes_)}
cat_mapping = {cat: int(code) for code, cat in enumerate(le_cat.classes_)}

# Robust Scaling
print("Fit RobustScaler trên tập train...")
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
    dtype='float32'
)

y_train_dict = {
    'label_encoded': y_train_enc,
    'label_name': y_train.values,
    'category_encoded': y_cat_train_enc,
    'category_name': y_cat_train.values,
    'is_attack': y_bin_train.values
}
print(f"Chuẩn hóa hoàn tất! X_train: {X_train_scaled.shape[0]:,} dòng x {X_train_scaled.shape[1]} đặc trưng.")

Fit RobustScaler trên tập train...
Chuẩn hóa hoàn tất! X_train: 4,114,778 dòng x 58 đặc trưng.


## Bước 6: Lưu Deliverables & Pipeline Tiền Xử Lý
Lưu trữ các artifacts vào `data/` và `models/`.

In [8]:
# ==============================================================================
# Cell 7: Lưu Trữ Deliverables
# ==============================================================================
# 1. Save Datasets
joblib.dump(X_train_scaled, 'data/X_train.pkl', compress=3)
joblib.dump(y_train_dict, 'data/y_train.pkl', compress=3)
print("Đã lưu: data/X_train.pkl & data/y_train.pkl")

# 2. Save Preprocessor Pipeline
preprocessor = {
    'scaler': scaler,
    'feature_names': X_train_scaled.columns.tolist(),
    'raw_feature_cols': feat_cols_raw,
    'train_row_hashes': train_row_hashes,
    'dropped_zero_var_cols': dropped_zero_var_cols,
    'imputation_medians': imputation_medians,
    'label_encoder': le_label,
    'category_encoder': le_cat,
    'label_mapping': label_mapping,
    'category_mapping': cat_mapping,
    'attack_category_mapping': ATTACK_CATEGORY_MAPPING,
    'engineered_features': [c for c in X_train_scaled.columns if c not in features or c not in DTYPE_DICT]
}
joblib.dump(preprocessor, 'models/preprocessor.joblib')

with open('models/label_mapping.json', 'w') as f:
    json.dump({'fine_grained_labels': label_mapping, 'attack_categories': cat_mapping}, f, indent=4)

print("Đã cập nhật: models/preprocessor.joblib & models/label_mapping.json")
print("\nPhase 2 hoàn thành xuất sắc!")

Đã lưu: data/X_train.pkl & data/y_train.pkl
Đã cập nhật: models/preprocessor.joblib & models/label_mapping.json

Phase 2 hoàn thành xuất sắc!
